In [66]:
import pandas as pd

from sklearn.model_selection import train_test_split

In [44]:
pd.set_option('display.max_columns', None)

TARGET_COL = "Customer Lifetime Value"

In [45]:
df = pd.read_csv("auto_insurance.csv")

In [46]:
df.head()

,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,Income,Location Code,Marital Status,Monthly Premium Auto,Months Since Last Claim,Months Since Policy Inception,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size
0,BU79786,Washington,2763.519279,No,Basic,Bachelor,2/24/11,Employed,F,56274,Suburban,Married,69,32,5,0,1,Corporate Auto,Corporate L3,Offer1,Agent,384.811147,Two-Door Car,Medsize
1,QZ44356,Arizona,6979.535903,No,Extended,Bachelor,1/31/11,Unemployed,F,0,Suburban,Single,94,13,42,0,8,Personal Auto,Personal L3,Offer3,Agent,1131.464935,Four-Door Car,Medsize
2,AI49188,Nevada,12887.431650,No,Premium,Bachelor,2/19/11,Employed,F,48767,Suburban,Married,108,18,38,0,2,Personal Auto,Personal L3,Offer1,Agent,566.472247,Two-Door Car,Medsize
3,WW63253,California,7645.861827,No,Basic,Bachelor,1/20/11,Unemployed,M,0,Suburban,Married,106,18,65,0,7,Corporate Auto,Corporate L2,Offer1,Call Center,529.881344,SUV,Medsize
4,HB64268,Washington,2813.692575,No,Basic,Bachelor,3/2/2011,Employed,M,43836,Rural,Single,73,12,44,0,1,Personal Auto,Personal L1,Offer1,Agent,138.130879,Four-Door Car,Medsize


In [47]:
print(f"Shape :\n {df.shape}")
print("-"*141)
print(f"Columns :\n {df.columns.to_list()}")
print("-"*141)
print(f"types :\n {df.dtypes}")
print("-"*141)
print(f"Missing values :\n {df.isna().sum()}")
print("-"*141)
print(f"duplicate : {df.duplicated().sum()}")
print("-"*141)
print(f"nunique : {df.nunique()}")

Shape :
 (9134, 24)
---------------------------------------------------------------------------------------------------------------------------------------------
Columns :
 ['Customer', 'State', 'Customer Lifetime Value', 'Response', 'Coverage', 'Education', 'Effective To Date', 'EmploymentStatus', 'Gender', 'Income', 'Location Code', 'Marital Status', 'Monthly Premium Auto', 'Months Since Last Claim', 'Months Since Policy Inception', 'Number of Open Complaints', 'Number of Policies', 'Policy Type', 'Policy', 'Renew Offer Type', 'Sales Channel', 'Total Claim Amount', 'Vehicle Class', 'Vehicle Size']
---------------------------------------------------------------------------------------------------------------------------------------------
types :
 Customer                          object
State                             object
Customer Lifetime Value          float64
Response                          object
Coverage                          object
Education                         obj

In [48]:
df[TARGET_COL].describe()

count     9134.000000
mean      8004.940475
std       6870.967608
min       1898.007675
25%       3994.251794
50%       5780.182197
75%       8962.167041
max      83325.381190
Name: Customer Lifetime Value, dtype: float64

In [49]:
print(f"Income == 0 : {(df["Income"] == 0).sum()}")

Income == 0 : 2317


In [50]:
pd.crosstab(df["EmploymentStatus"], df["Income"] == 0)

Income,False,True
EmploymentStatus,,
Disabled,405,0
Employed,5698,0
Medical Leave,432,0
Retired,282,0
Unemployed,0,2317


In [51]:
df.loc[df["Income"] > 0, 'Income'].describe()

count     6817.000000
mean     50456.580461
std      24305.956566
min      10037.000000
25%      28432.000000
50%      48081.000000
75%      70187.000000
max      99981.000000
Name: Income, dtype: float64

In [52]:
parsed = pd.to_datetime(df["Effective To Date"], errors='coerce')

/var/folders/64/grj5pyf15ys44prkvy9xkwbr0000gn/T/ipykernel_29432/812662837.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df["Effective To Date"], errors='coerce')


In [53]:
print("Failed to parse:", parsed.isna().sum())
print("Min date:", parsed.min())
print("Max date:", parsed.max())
print("-"*75)
print("Rows per month:")
print(parsed.dt.month.value_counts().sort_index())
print("-"*75)
print(df.loc[[9129, 9130, 9131, 9132], "Effective To Date"])

Failed to parse: 0
Min date: 2011-01-01 00:00:00
Max date: 2011-12-02 00:00:00
---------------------------------------------------------------------------
Rows per month:
Effective To Date
1     3356
2     2722
3      336
4      279
5      325
6      287
7      308
8      283
9      283
10     356
11     317
12     282
Name: count, dtype: int64
---------------------------------------------------------------------------
9129    10/2/2011
9130    12/2/2011
9131     6/2/2011
9132     3/2/2011
Name: Effective To Date, dtype: object


In [54]:
raw_date = df["Effective To Date"]

year_len = raw_date.str.split("/").str[2].str.len()
print(year_len.value_counts())
print("-"*75)
short = raw_date[year_len == 2]
long = raw_date[year_len == 4]

print("Short, m/d/y:",
      pd.to_datetime(short, format="%m/%d/%y").min(),
      pd.to_datetime(short, format="%m/%d/%y").max())

print("-"*75)

long_md = pd.to_datetime(long, format="%m/%d/%Y")
print("Long, m/d/Y:", long_md.min(), long_md.max())
print("-"*75)

long_dm = pd.to_datetime(long, format="%d/%m/%Y")
print("Long, d/m/Y:", long_dm.min(), long_dm.max())

Effective To Date
2    5467
4    3667
Name: count, dtype: int64
---------------------------------------------------------------------------
Short, m/d/y: 2011-01-13 00:00:00 2011-02-28 00:00:00
---------------------------------------------------------------------------
Long, m/d/Y: 2011-01-01 00:00:00 2011-12-02 00:00:00
---------------------------------------------------------------------------
Long, d/m/Y: 2011-01-01 00:00:00 2011-02-12 00:00:00


In [55]:
long_mask = year_len == 4
df["Effective To Date (parsed)"] = pd.NaT

df.loc[long_mask, "Effective To Date (parsed)"] = pd.to_datetime(long, format="%d/%m/%Y")
df.loc[~long_mask, "Effective To Date (parsed)"] = pd.to_datetime(short, format="%m/%d/%y")

In [56]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Customer Lifetime Value,9134.0,8004.940475,1898.007675,3994.251794,5780.182197,8962.167041,83325.38119,6870.967608
Income,9134.0,37657.380009,0.0,0.0,33889.5,62320.0,99981.0,30379.904734
Monthly Premium Auto,9134.0,93.219291,61.0,68.0,83.0,109.0,298.0,34.407967
Months Since Last Claim,9134.0,15.097,0.0,6.0,14.0,23.0,35.0,10.073257
Months Since Policy Inception,9134.0,48.064594,0.0,24.0,48.0,71.0,99.0,27.905991
Number of Open Complaints,9134.0,0.384388,0.0,0.0,0.0,0.0,5.0,0.910384
Number of Policies,9134.0,2.96617,1.0,1.0,2.0,4.0,9.0,2.390182
Total Claim Amount,9134.0,434.088794,0.099007,272.258244,383.945434,547.514839,2893.239678,290.500092
Effective To Date (parsed),9134,2011-01-29 20:06:21.519597056,2011-01-01 00:00:00,2011-01-15 00:00:00,2011-01-29 00:00:00,2011-02-13 00:00:00,2011-02-28 00:00:00,NaN


In [58]:
col = "Effective To Date (parsed)"

print("Rows still NaT:", df[col].isna().sum())
print("Min:", df[col].min(), "| Max:", df[col].max())
print("Unique dates:", df[col].nunique())
print("Rows:", len(df))

Rows still NaT: 0
Min: 2011-01-01 00:00:00 | Max: 2011-02-28 00:00:00
Unique dates: 59
Rows: 9134


In [60]:
rows_before = len(df)
print(f"Rows Before : {rows_before}")

print("duplicates :", df.duplicated().sum())

df_noid = df.drop(columns=["Customer"])

print("Duplicates AFTER dropping the ID :", df_noid.duplicated().sum())

Rows Before : 9134
duplicates : 0
Duplicates AFTER dropping the ID : 163


In [63]:
df_noid[df_noid.duplicated(keep=False)].sort_values(by=TARGET_COL).head(10)

,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,Income,Location Code,Marital Status,Monthly Premium Auto,Months Since Last Claim,Months Since Policy Inception,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Effective To Date (parsed)
7135,Oregon,2004.350666,Yes,Basic,College,2/22/11,Unemployed,F,0,Suburban,Divorced,66,16,35,4,1,Personal Auto,Personal L3,Offer1,Branch,316.800000,Four-Door Car,Small,2011-02-22
3665,Oregon,2004.350666,Yes,Basic,College,2/22/11,Unemployed,F,0,Suburban,Divorced,66,16,35,4,1,Personal Auto,Personal L3,Offer1,Branch,316.800000,Four-Door Car,Small,2011-02-22
4104,Oregon,2300.691547,Yes,Basic,Bachelor,1/15/11,Retired,M,25905,Suburban,Divorced,63,21,12,0,1,Personal Auto,Personal L3,Offer1,Agent,302.400000,Four-Door Car,Large,2011-01-15
4502,Oregon,2300.691547,Yes,Basic,Bachelor,1/15/11,Retired,M,25905,Suburban,Divorced,63,21,12,0,1,Personal Auto,Personal L3,Offer1,Agent,302.400000,Four-Door Car,Large,2011-01-15
5167,Nevada,2359.423599,Yes,Basic,Bachelor,11/1/2011,Disabled,F,19864,Suburban,Divorced,63,22,96,0,1,Personal Auto,Personal L3,Offer1,Web,352.394515,Four-Door Car,Medsize,2011-01-11
6414,Nevada,2359.423599,Yes,Basic,Bachelor,11/1/2011,Disabled,F,19864,Suburban,Divorced,63,22,96,0,1,Personal Auto,Personal L3,Offer1,Web,352.394515,Four-Door Car,Medsize,2011-01-11
5632,Oregon,2395.570000,Yes,Basic,Doctor,1/27/11,Disabled,F,21986,Suburban,Divorced,63,1,9,0,1,Personal Auto,Personal L3,Offer2,Branch,333.696228,Two-Door Car,Medsize,2011-01-27
4935,Oregon,2395.570000,Yes,Basic,Doctor,1/27/11,Disabled,F,21986,Suburban,Divorced,63,1,9,0,1,Personal Auto,Personal L3,Offer2,Branch,333.696228,Two-Door Car,Medsize,2011-01-27
1774,California,2396.219292,Yes,Basic,Bachelor,10/1/2011,Unemployed,F,0,Suburban,Married,66,6,71,2,1,Corporate Auto,Corporate L1,Offer2,Call Center,433.160284,Four-Door Car,Large,2011-01-10
6586,California,2396.219292,Yes,Basic,Bachelor,10/1/2011,Unemployed,F,0,Suburban,Married,66,6,71,2,1,Corporate Auto,Corporate L1,Offer2,Call Center,433.160284,Four-Door Car,Large,2011-01-10


In [65]:
df_clean = df_noid.drop_duplicates()

print("Rows before:", rows_before)
print("Rows after:", len(df_clean))
print("Rows removed:", rows_before - len(df_clean))
print("Shape:", df_clean.shape)
print("Duplicates left:", df_clean.duplicated().sum())

Rows before: 9134
Rows after: 8971
Rows removed: 163
Shape: (8971, 24)
Duplicates left: 0


In [69]:
X = df_clean.drop(columns=[TARGET_COL])
y = df_clean[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Shapes:")
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)
print('-'*75)

print(TARGET_COL in X_train.columns.to_list())
print(TARGET_COL in X_test.columns.to_list())

print('-'*75)
print("y_train :")
print(y_train.describe())
print('-'*50)
print("y_test :")
print(y_test.describe())

Shapes:
(7176, 23)
(1795, 23)
(7176,)
(1795,)
---------------------------------------------------------------------------
False
False
---------------------------------------------------------------------------
y_train :
count     7176.000000
mean      8008.349211
std       6917.658715
min       1898.007675
25%       3985.490513
50%       5802.450724
75%       8984.682679
max      74228.516040
Name: Customer Lifetime Value, dtype: float64
--------------------------------------------------
y_test :
count     1795.000000
mean      7987.574014
std       6667.337637
min       2034.993043
25%       4022.239978
50%       5717.221919
75%       8900.944435
max      83325.381190
Name: Customer Lifetime Value, dtype: float64
